# Notebook 3: Extracción y Preprocesamiento - Guías CANMAT (Trastornos del Estado de Ánimo)

Este notebook procesa las guías clínicas de la **CANMAT** (*Canadian Network for Mood and Anxiety Treatments* / ISBD) para **Depresión Mayor** y **Trastorno Bipolar**.

### Objetivos:
1. Extraer las recomendaciones clínicas estructuradas por jerarquía terapéutica (Primera Línea, Segunda Línea y tratamientos No Recomendados).
2. Capturar los **Niveles de Evidencia científica** (Level 1, Level 2, y la etiqueta propia 'Level 1 (Daño)' para recomendaciones en contra) asignados por los consensos internacionales.
3. Modelar las reglas de prescripción y evitar tratamientos contraindicados (ej. evitar antidepresivos en monoterapia en depresión bipolar).
4. Generar la base de conocimiento clínico que alimentará el motor de recomendaciones y el RAG en Streamlit.


In [1]:
import sys
from pathlib import Path
import pandas as pd
import re

# Detectar la ruta raíz del proyecto de forma dinámica y robusta
def _encontrar_raiz():
    cwd = Path.cwd().resolve()
    for p in [cwd] + list(cwd.parents):
        if (p / 'src').is_dir() and (p / 'data').is_dir():
            return p
        if (p / 'Sofia' / 'src').is_dir() and (p / 'Sofia' / 'data').is_dir():
            return p / 'Sofia'
    raise RuntimeError("No se pudo determinar la raíz del proyecto Sofia.")

project_root = _encontrar_raiz()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.extractors.canmat_extractor import CANMATExtractor

DATA_RAW = project_root / 'data' / 'raw'
DATA_PROCESSED = project_root / 'data' / 'processed'
DATA_PROCESSED.mkdir(parents=True, exist_ok=True)

print('Entorno de CANMAT configurado. Directorio de datos: data/processed')


Entorno configurado correctamente. Directorio de datos: data/processed


## 1. Corpus de Recomendaciones CANMAT

Estructuramos las tablas de evidencia de las guías CANMAT (consenso de adultos para depresión y bipolaridad).


In [2]:
# Verificación del archivo oficial CANMAT/ISBD 2018 (BDI-20-97.pdf para Bipolar) y CANMAT 2016 (para TDM)
# Nota metodológica: BDI-20-97.pdf corresponde al consenso CANMAT/ISBD 2018 para Trastorno Bipolar,
# mientras que las directrices de Depresión Unipolar provienen del consenso CANMAT 2016.
canmat_extractor = CANMATExtractor(raw_dir=DATA_RAW)
pdf_path = canmat_extractor.find_pdf_path()

if pdf_path:
    print(f'PDF oficial de CANMAT Bipolar 2018 localizado: {pdf_path.name} ({pdf_path.stat().st_size:,} bytes)')
else:
    print('No se detectó el PDF local; utilizando corpus de directrices clínicas validadas (CANMAT 2018 Bipolar + CANMAT 2016 TDM).')

# Extracción de recomendaciones completas y alertas clínicas
df_canmat, df_alertas_canmat = canmat_extractor.extract_recommendations()

print(f'Total de directrices CANMAT procesadas: {len(df_canmat)}')
df_canmat.head(8)


PDF oficial de CANMAT Bipolar 2018 localizado: BDI-20-97.pdf (3,590,211 bytes)
Total de directrices CANMAT procesadas: 22


,patologia,fase,linea,estrategia,farmaco,atc,nivel_evidencia,comentarios
0,Trastorno Depresivo Mayor (TDM),Aguda,Primera Línea,Monoterapia,Citalopram,N06AB04,Level 1,ISRS de alta tolerabilidad; vigilar intervalo ...
1,Trastorno Depresivo Mayor (TDM),Aguda,Primera Línea,Monoterapia,Escitalopram,N06AB10,Level 1,ISRS con menor perfil de interacciones citocro...
2,Trastorno Depresivo Mayor (TDM),Aguda,Primera Línea,Monoterapia,Sertralina,N06AB06,Level 1,ISRS seguro en cardiopatía isquémica (estudio ...
3,Trastorno Depresivo Mayor (TDM),Aguda,Primera Línea,Monoterapia,Venlafaxina,N06AX16,Level 1,IRSN dual; monitorizar presión arterial a dosi...
4,Trastorno Depresivo Mayor (TDM),Aguda,Primera Línea,Monoterapia,Duloxetina,N06AX21,Level 1,IRSN indicado en depresión con dolor somático ...
5,Trastorno Depresivo Mayor (TDM),Aguda,Primera Línea,Monoterapia,Bupropión,N06AX12,Level 1,IRND sin disfunción sexual ni ganancia pondera...
6,Trastorno Depresivo Mayor (TDM),Aguda,Primera Línea,Monoterapia,Vortioxetina,N06AX26,Level 1,Modulador multimodal serotonérgico con benefic...
7,Trastorno Depresivo Mayor (TDM),Aguda,Segunda Línea,Monoterapia,Amitriptilina,N06AA09,Level 1,Tricíclico eficaz pero con alta carga anticoli...


## 2. Limpieza, Validación y Normalización de Categorías

Estandarizamos los textos y validamos las clasificaciones de evidencia para asegurar consistencia con el sistema ATC.


In [3]:
# Limpieza de espacios en ambas tablas (df_canmat y df_alertas_canmat)
for df in [df_canmat, df_alertas_canmat]:
    for c in [col for col in df.columns if pd.api.types.is_string_dtype(df[col])]:
        df[c] = df[c].str.strip()

# Comprobación de nulos y duplicados en columnas clave
cols_clave = ['patologia', 'linea', 'estrategia', 'farmaco', 'nivel_evidencia']
nulos_clave = df_canmat[cols_clave].isnull().sum().to_dict()
print(f"Control de calidad en columnas clave: {nulos_clave}")
duplicados = df_canmat.duplicated().sum()
print(f"Duplicados en df_canmat: {duplicados}")
assert duplicados == 0, f"Error: Hay {duplicados} filas duplicadas en df_canmat"

# Comprobación de formato ATC nivel 5
atc_val = df_canmat['atc'].dropna()
atc_regex = re.compile(r'^[A-Z]\d{2}[A-Z]{2}\d{2}$')
atc_invalidos = atc_val[~atc_val.apply(lambda x: bool(atc_regex.match(str(x))))].unique().tolist()
if atc_invalidos:
    print(f"ℹ Aviso: Códigos ATC no estándar (agrupaciones o combinaciones clínicas): {atc_invalidos}")

# Valores únicos de línea y nivel_evidencia
print(f"Valores únicos de Línea: {df_canmat['linea'].unique().tolist()}")
print(f"Valores únicos de Nivel de Evidencia: {df_canmat['nivel_evidencia'].unique().tolist()}")

# Tabla cruzada de contingencia original
print("\nDistribución por Patología y Nivel de Evidencia:")
df_canmat.groupby(['patologia', 'nivel_evidencia']).size().unstack(fill_value=0)


Distribución por Patología y Nivel de Evidencia:


nivel_evidencia,Level 1,Level 1 (Daño),Level 2
patologia,,,
Trastorno Bipolar,6,1,3
Trastorno Depresivo Mayor (TDM),11,0,1


## 3. Generación de Reglas Clínicas para el Motor de Cascadas y RAG

Construimos una tabla derivada con alertas clínicas directas que se utilizarán para validar combinaciones potencialmente peligrosas o inapropiadas.


In [4]:
# Visualización de directrices con alertas críticas (Riesgo de Viraje a Manía o Contraindicación)
print(f'Total de alertas clínicas generadas: {len(df_alertas_canmat)}')
df_alertas_canmat


Total de alertas clínicas generadas: 1


,patologia,farmaco_involucrado,tipo_alerta,descripcion,nivel_evidencia,fuente
0,Trastorno Bipolar,Antidepresivos ISRS en Monoterapia,No Recomendado / Riesgo de Viraje,ALERTA: No recomendados por alto riesgo de vir...,Level 1 (Daño),Guías CANMAT / ISBD


## 4. Exportación de Resultados

Guardamos las recomendaciones estructuradas de CANMAT en la carpeta de datos procesados.


In [5]:
# Verificación estricta del esquema de 8 columnas antes de exportar
columnas_esperadas = ['patologia', 'fase', 'linea', 'estrategia', 'farmaco', 'atc', 'nivel_evidencia', 'comentarios']
for col in columnas_esperadas:
    assert col in df_canmat.columns, f"Error: Falta la columna requerida '{col}' en df_canmat"

out_canmat = DATA_PROCESSED / 'canmat_recomendaciones_clean.csv'
out_alertas = DATA_PROCESSED / 'canmat_alertas_clinicas_clean.csv'

df_canmat.to_csv(out_canmat, index=False, encoding='utf-8')
df_alertas_canmat.to_csv(out_alertas, index=False, encoding='utf-8')

rel_canmat = out_canmat.relative_to(project_root)
rel_alertas = out_alertas.relative_to(project_root)
print(f'Recomendaciones CANMAT guardadas en: {rel_canmat} ({len(df_canmat)} filas)')
print(f'Alertas clínicas CANMAT guardadas en: {rel_alertas} ({len(df_alertas_canmat)} filas)')


Recomendaciones CANMAT guardadas en: data/processed/canmat_recomendaciones_clean.csv (22 filas)
Alertas clínicas CANMAT guardadas en: data/processed/canmat_alertas_clinicas_clean.csv (1 filas)
